In [6]:
from diagrams import Diagram, Cluster, Edge
from diagrams.onprem.client import User
from diagrams.programming.framework import React, Spring
from diagrams.onprem.database import PostgreSQL
from diagrams.onprem.inmemory import Redis
from diagrams.onprem.queue import Kafka
from diagrams.aws.storage import S3
from diagrams.onprem.security import Vault
from diagrams.onprem.container import Docker

with Diagram(
    " ",
    filename="citymitra_detailed_architecture",
    outformat="png",
    show=False,
    direction="LR"
):

    # ================= ACTORS =================
    citizen = User("Citizen\n(USER)")
    admin = User("Admin")
    maintainer = User("Maintainer")

    # ================= FRONTEND =================
    with Cluster("Presentation Layer"):
        react = React("React Web App\n(Map + Dashboard)")

    # ================= BACKEND =================
    with Cluster("Application Layer"):
        spring = Spring("Spring Boot API\n(Auth, Issues, SLA)")
        security = Vault("JWT")

    # ================= CACHING =================
    with Cluster("Caching Layer"):
        redis = Redis("Redis\n(Session + Issue Cache)")

    # ================= EVENT STREAMING =================
    with Cluster("Event-Driven Layer"):
        kafka = Kafka("Kafka\n(Issue & SLA Events)")

    # ================= DATA =================
    with Cluster("Persistence Layer"):
        postgres = PostgreSQL(
            "PostgreSQL\nUsers • Roles\nIssues • History"
        )
        s3 = S3(
            "AWS S3\nIssue Images\nResolution Proof"
        )

    # ================= FLOWS =================
    citizen >> react
    admin >> react
    maintainer >> react

    react >> spring
    spring >> security

    spring >> Edge(label="Read/Write") >> postgres
    spring >> Edge(label="Cache") >> redis
    redis >> Edge(label="Cache Hit") >> spring

    spring >> Edge(label="Publish Events") >> kafka
    kafka >> Edge(label="Consume Events") >> spring

    spring >> Edge(label="Store Files") >> s3
